In [ ]:
import openai

# Paste your existing API key here
openai.api_key = "OPENAI_API_KEY"  # 👈 Replace this with your actual key

try:
    models = openai.models.list()
    print("✅ API Key is working! Available models:")
    for m in models.data[:5]:
        print("-", m.id)
except Exception as e:
    print("❌ API Key is invalid or not working.")
    print("Error:", e)


✅ API Key is working! Available models:
- text-embedding-ada-002
- whisper-1
- gpt-3.5-turbo
- tts-1
- gpt-3.5-turbo-16k


In [3]:
pip install openai faiss-cpu transformers sentence-transformers pandas


Note: you may need to restart the kernel to use updated packages.


ERROR: Error while checking for conflicts. Please file an issue on pip's issue tracker: https://github.com/pypa/pip/issues/new
Traceback (most recent call last):
  File "C:\Users\japje\anaconda3\Lib\site-packages\pip\_internal\commands\install.py", line 587, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\japje\anaconda3\Lib\site-packages\pip\_internal\operations\check.py", line 116, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\japje\anaconda3\Lib\site-packages\pip\_internal\operations\check.py", line 58, in create_package_set_from_installed
    package_set[name] = PackageDetails(dist.version, dependencies)
                                       ^^^^^^^^^^^^
  File "C:\Users\japje\anaconda3\Lib\site-packages\pip\_internal\metadata\importlib\_dists.py", line 175, in version
    return parse

In [ ]:
# ✅ Imports
import os
import pandas as pd
import numpy as np
import faiss
import torch
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ✅ Load CSV
csv_path = r"C:\Users\japje\Documents\B.G\wiki_hybrid_chunks_300.csv"
df = pd.read_csv(csv_path)
texts = df["chunk_text"].tolist()
id_to_text = df[["chunk_id", "chunk_text"]].reset_index(drop=True)

# ✅ Embedding
print("🔄 Encoding document chunks...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
dim = embeddings.shape[1]

# ✅ FAISS Index
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# ✅ Load TinyLLaMA model (fallback)
print("🔧 Loading TinyLLaMA model (fallback)...")
tinyllama_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(tinyllama_model_id)
model = AutoModelForCausalLM.from_pretrained(tinyllama_model_id)
local_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)

# ✅ OpenAI Setup
openai_api_key = "OPENAI_API_KEY"  # Replace with your valid API key
client = OpenAI(api_key=openai_api_key)
use_openai = True
k = 5  # Top-K chunks

# ✅ RAG Ask Function
def ask_question(query):
    # 🔍 Retrieve chunks from FAISS
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [id_to_text.iloc[idx]["chunk_text"] for idx in indices[0]]
    
    # ✂️ Truncate long context to fit token limit
    context = "\n".join(retrieved_chunks[:3])
    max_context_len = 1500
    context = context[:max_context_len]

    # 🧠 Prompt
    prompt = f"""You are a helpful assistant. Use the following context to answer the question:

Context:
{context}

Question: {query}
Answer:"""

    # 🧠 Try OpenAI GPT-3.5
    answer = None
    if use_openai:
        try:
            print("🔁 Trying OpenAI GPT-3.5...")
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=500
            )
            answer = response.choices[0].message.content
            print("✅ Answer from OpenAI GPT-3.5:\n")
        except Exception as e:
            print(f"⚠️ OpenAI failed: {e}")
            print("🔁 Falling back to TinyLLaMA...\n")

    # 🔁 TinyLLaMA Fallback
    if answer is None:
        output = local_pipe(prompt, max_new_tokens=300, temperature=0.7, do_sample=True)
        answer = output[0]["generated_text"].split("Answer:")[-1].strip()
        print("✅ Answer from TinyLLaMA:\n")

    # 📄 Show context
    print("\n🔎 Retrieved Chunks:\n")
    for idx in indices[0]:
        print(f"- {id_to_text.iloc[idx]['chunk_text']}\n---")

    # 💬 Final Answer
    print("\n💬 Final Answer:\n", answer)

# ✅ Example usage
ask_question("who is bob brown")


🔄 Encoding document chunks...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

🔧 Loading TinyLLaMA model (fallback)...


Device set to use cpu


🔁 Trying OpenAI GPT-3.5...
⚠️ OpenAI failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
🔁 Falling back to TinyLLaMA...

✅ Answer from TinyLLaMA:


🔎 Retrieved Chunks:

- for other people with the same name, see rob brown. robert william brown born april 10, 1968 is a canadian former professional ice hockey right winger. rob brown he is best known for his time spent playing for the pittsburgh penguins from his debut in 1987 until 1990, and then again from 1997 until 2000. between and following these stints, brown shuffled between minor league teams in the international hockey league ihl and other nhl teams, including the hartford whalers, chicago blackhawks, dallas stars, and los angeles kings. playing career as a youth, he